# 12 — Donors and the inference machinery

**Stage question.** What do N1 and N2 preserve and destroy, and does the session bootstrap pass its self-test?

| | |
|---|---|
| **Purpose** | Draw N1 and N2 donors, validate every registered property, show matching quality, and run the bootstrap self-test. No real Δ is computed here: calibration (notebook 14) comes first. |
| **Input** | The analysis data. |
| **Processing** | seeded donor draws → validity assertions → matching report; bootstrap self-test on simulated clustered data. |
| **Output** | `results/donors/` reports. |
| **Scientific assumptions** | N1 matched on duration decile and word count; N2 inside the interaction at ≥ 10 turns. |
| **Failure modes** | A pair with no admissible donor (raises). |
| **What to look at** | The exact-match share for N1 and the self-test checks. |

**Terms used in this notebook.**

- **Donor** — a turn whose delivery vector replaces the partner's in a null refit.
- **Cluster bootstrap** — resampling whole sessions with replacement.

> **Before running.** The bootstrap settings are in `src/settings.py`.

In [1]:
# Runtime setup — identical in every notebook.
# Colab: mounts Google Drive and uses MyDrive/coda (a copy of the repository) as
#        the project root, then installs the pinned measurement stack.
# Local: uses the repository checkout.
# Every path used below is relative to that root, so Drive and the repository
# have exactly the same layout and outputs can be copied back unchanged.
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    ROOT = Path(os.environ.get('CODA_ROOT', '/content/drive/MyDrive/coda'))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    str(ROOT / 'requirements-colab.txt')], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

os.environ['CODA_ROOT'] = str(ROOT)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.runtime import runtime_report
REPORT = runtime_report()

Mounted at /content/drive
runtime        colab   python 3.13.16   cpus 2
memory         12.4 GB free of 13.6 GB
gpu            none (not needed for this stage unless stated)
project disk   11.6 GB free   root = /content/drive/MyDrive/coda
scratch disk   93.4 GB free   (audio lives here briefly)
code           fingerprint 9b5146f3ab46b0fc
artifacts      manifest=True, metadata=True, transcripts=True, turns=True, audio_pass_sessions=293, channel_qc=True


**Runtime check.** The cell above prints free disk, memory, CPU, GPU, the code fingerprint and which artifacts already exist. The fingerprint identifies the exact source and configuration that ran and is written into every run record.

In [2]:
STAGE = 'sample'
N_DRAWS = 50
STEPS = ['donors']

## Settings

The method settings come from `src/settings.py` (class `Settings`).

In [3]:
from src.settings import Settings
SETTINGS = Settings()

In [4]:
from src.paths import paths
from src.experiments.stages import analysis_data
from src.experiments.run_j2 import draw_donors
P = paths()
data, turns, prosody, split = analysis_data(P, STAGE, 'speaker_disjoint', SETTINGS)
d = draw_donors(data, turns, prosody, N_DRAWS, seed=20260912)
rep = d['n1_report']
print('N1 exact word-count match share:', rep['exact_word_match'].mean().round(3))
rep.groupby('split')['candidates'].describe()

/usr/local/lib/python3.13/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/content/drive/MyDrive/coda/src/data/dataset.py:180: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False).astype(bool),


N1 exact word-count match share: 0.997


,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,12107.0,305.984885,329.026041,1.0,57.0,173.0,396.5,1070.0
train,37600.0,964.994548,1068.496304,1.0,177.0,434.0,1268.0,3347.0
val,6922.0,178.000144,204.716735,1.0,27.0,80.0,231.0,640.0


**Reading it.** Every assignment has already passed the validity assertions (donor never the true partner, same split, N1 from another session, N2 same session, speaker and interaction at ten or more turns). The exact-match share says how often N1 matched word count exactly rather than by the nearest count in the same duration decile.

In [5]:
from src.evaluation.bootstrap import self_test
self_test(seed=0)

{'passed': True,
 'checks': {'resamples_are_whole_sessions': True,
  'se_within_15pct_of_analytic': True,
  'pair_level_bootstrap_understates_se': True,
  'coverage_95_within_3_mc_se': np.True_},
 'se_bootstrap': 0.1236512642839137,
 'se_analytic': 0.12868698739710885,
 'se_pair_level': 0.017434327903027517,
 'coverage_95': np.float64(0.92),
 'n_reps': 200}

**Reading it.** On simulated clustered data the session bootstrap's standard error matches the analytic cluster standard error, a pair-level bootstrap understates it, and 95% intervals cover at close to the nominal rate. Coverage with about fifty test sessions can sit a little below 95%; that is reported beside every interval.